In [1]:
import pathlib

import yaml
import scipy.signal
import pandas as pd
import numpy as np

# import jax
# import jax.numpy as jnp
import jax.random as jr
from jaxtyping import Array, Float

import torch
from torch.utils.data import DataLoader

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core
import trapallada.plot

# import ipywidgets as widgets
# from sklearn.manifold import TSNE

# TODO

- normalize the signals so that they are between 0 and 1, and then `tanh` en el codificador

So that `impor`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory containing the data

In [3]:
data_directory = pathlib.Path.cwd().parent / 'data'
assert data_directory.exists()

The particular dataset (specified by a *base* file name)

In [4]:
# base_filename = '4_classes_modulated'
base_filename = '20_600'

signals_file = data_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

# metadata_file = data_directory / f'{base_filename}_labels.parquet'
metadata_file = data_directory / f'{base_filename}_metadata.parquet'
assert metadata_file.exists()

# signals_file, metadata_file

Dataset is loaded

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
# len(ds)

trapallada.data - WARNING - dismissed 634 constant signals (that cannot be normalized); 43669 left


The directory containing the actual results

In [6]:
# results_directory = pathlib.Path.cwd()
results_directory = pathlib.Path.cwd() / 'results'
assert results_directory.exists()

In [7]:
list(results_directory.glob('*.eqx'))

[PosixPath('/home/manu/Sync/git/trapallada/experiments/results/real_z_20_4000_epochs.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/smaller.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/real_z_50_4000_epochs.eqx')]

*Base* file name for the results (trained model)

In [8]:
# input_files_stem = pathlib.Path('/tmp') / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_500e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z20_300e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_10_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_50_300_epochs' # <-------------------------------------
# input_files_stem = results_directory / 'new' / 'z_2_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_3_300_epochs' 
# input_files_stem = results_directory / 'new' / 'real_z_50_300_epochs'
# ---
input_files_stem = results_directory / 'real_z_50_4000_epochs'
# input_files_stem = results_directory / 'real_z_20_4000_epochs'
# input_files_stem = results_directory / 'smaller'


In [9]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [10]:
# w_file_upload.value

From the above *stem*, several file names are derived

In [11]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [12]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
# parameters
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
encoder:
  input_size:
  - 1
  - 600
loss:
  x_sd: 0.25
model:
  channels:
  - 12
  - 12
  - 12
  dim_z: 50 
  kernel_sizes:
  - 128
  - 64
  - 32
  stride: 2
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 4000
  train_test_split:
    test_size: 0.2


In [13]:
results_df = pd.read_parquet(results_file)
results_df

training                             validation                \
             loss loglikelihood          kl         loss loglikelihood   
0     4576.396764  -4558.805352   17.591412  4530.477217  -4514.667282   
1     4523.245949  -4513.040764   10.205187  4516.150448  -4512.856726   
2     4514.254346  -4512.674148    1.580197  4513.288522  -4512.441272   
3     4512.742236  -4512.010177    0.732065  4512.028233  -4511.164883   
4     4497.214746  -4493.381565    3.833186  4487.098706  -4481.943270   
...           ...           ...         ...          ...           ...   
2306  3660.148979  -3559.516712  100.632267  3826.042910  -3725.190054   
2307  3660.321094  -3559.769093  100.552002  3820.286071  -3718.370165   
2308  3660.244208  -3559.720237  100.523971  3820.072355  -3722.897861   
2309  3660.561598  -3559.837579  100.724023  3822.037789  -3718.380679   
2310  3660.013837  -3559.486989  100.526848  3819.097217  -3719.628210   

                  
              kl  
0      15.809943  
1       3.293721  
2       0.847262  
3       0.863352  
4       5.155438  
...          ...  
2306  100.852852  
2307  101.915902  
2308   97.174489  
2309  103.657111  
2310   99.469014  

[2311 rows x 6 columns]

# Training curves

In [14]:
def metric_chart(df: pd.DataFrame, metric: str):

    sub_df = df.loc[:,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

alt.HConcatChart(...)

In [15]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [16]:
sum(trapallada.util.get_layers_n_parameters(model).values())

97841

Data loaders for both the training and validation sets

In [17]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])

A batch from the **validation** set

In [18]:
# iter_dl = iter(train_dl) # training
iter_dl = iter(validation_dl) # validation
next(iter_dl)
batch = next(iter_dl)

Is every class present?

In [19]:
batch[2].unique()

tensor([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan], dtype=torch.float64)

In [20]:
# ds.metadata_df

An element from the above batch is encoded, decoded, and a prediction plotted

In [25]:
# i = 0
# i = 1
i = 25

# indexes from the above batch
i_s, *_ = batch

def plot_prediction(ds, i):
    
    x = ds[i][1].numpy()
    x_pred, z_mean, z_sd = model(x, jr.PRNGKey(42))
    display(ds.metadata_df.loc[i])
    df = pd.DataFrame({'prediction': x_pred[0], 'actual': x.ravel()})
    n_rows = 2000
    
    return trapallada.plot.df(df.iloc[:n_rows])

# plot_prediction(ds, 25)
plot_prediction(ds, i_s[i].item())

start            2007-03-19 17:42:04.176057
end              2007-03-19 22:33:34.976057
tar file                     0401870501.tar
source      P0401870501PNS003SRCTSR8001.FTZ
Name: 10650, dtype: object

alt.Chart(...)

## Javi's paper

In [27]:
javi_df = ds.metadata_df[ds.metadata_df['source'] == 'P0605950101PNS003SRCTSR8001.FTZ']
javi_df.tail()

,start,end,tar file,source
17450,2009-06-03 14:40:22.717454,2009-06-03 19:31:53.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
17454,2009-06-03 15:04:42.717454,2009-06-03 19:56:13.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
17458,2009-06-03 15:29:02.717454,2009-06-03 20:20:33.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
17462,2009-06-03 15:53:22.717454,2009-06-03 20:44:53.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
17466,2009-06-03 16:12:50.717454,2009-06-03 21:04:21.517454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ


In [28]:
(javi_df['end'] - javi_df['start']).tail()

17450   0 days 04:51:30.800000
17454   0 days 04:51:30.800000
17458   0 days 04:51:30.800000
17462   0 days 04:51:30.800000
17466   0 days 04:51:30.800000
dtype: timedelta64[ns]

In [29]:
n_signals_to_plot = 1
trapallada.plot.signals_array(ds[javi_df.tail(n_signals_to_plot).index][1].numpy().squeeze(axis=1))
# ds[javi_df.tail(n_signals_to_plot).index][1].shape

alt.Chart(...)

In [30]:
plot_prediction(ds, 17466)

start            2009-06-03 16:12:50.717454
end              2009-06-03 21:04:21.517454
tar file                     0605950101.tar
source      P0605950101PNS003SRCTSR8001.FTZ
Name: 17466, dtype: object

alt.Chart(...)

# Latent space projection

In [ ]:
df = trapallada.util.make_projections_df(ds, validation_dl, model.encoder)
# df = trapallada.util.make_projections_df(ds, train_dl, model.encoder)
df.head()

In [ ]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='tar file:N',
    fill='tar file:N',
    tooltip=['source:N']
    # tooltip=['tar file:N']
).properties(
    width=800, height=600
).interactive()

Signals that are just noise have no SNR

In [ ]:
df['SNR'].isna().sum()

As a *fix*, their SNR is set to the maximum

In [ ]:
df.loc[df['label'] == 'noise', 'SNR'] = df['SNR'].max()

...plotting

In [ ]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='label:N',
    fill='label:N',
    size='SNR:Q',
    tooltip=['file name:N']
).properties(
    width=800, height=600
).interactive()

The latent space is projected into a 3D space

In [ ]:
df_3 = trapallada.util.make_projections_df(ds, validation_dl, model.encoder, n_projected_components=3)
df_3.head()

In [ ]:
import plotly.express as px
fig = px.scatter_3d(df_3, x='tsne_1', y='tsne_2', z='tsne_3', color='label', opacity=0.5, width=800, height=800)
fig.show()

# Real data

In [ ]:
# signals_file = data_directory / '0605950101.pt'
signals_file = data_directory / '0605950101_bin_size_3.pt'
assert signals_file.exists()

In [ ]:
real_ds = trapallada.data.SignalsDataset(signals_file, metadata_filename=None, normalize=True)

In [ ]:
real_ds[3:5]

Data loaders for both the training and validation sets

In [ ]:
torch_rng = torch.Generator().manual_seed(parameters['PRNG seeds']['training dataloader'])
real_dl = DataLoader(real_ds, parameters['training']['batch_size'], shuffle=True, generator=torch_rng)

A batch

In [ ]:
batch = next(iter(real_dl))
batch

In [ ]:
signals = batch[1]
signals.shape

Let us drop the last few measurements so that the signals have the appropriate length for the trained model

In [ ]:
signals = signals[..., :10_000]
signals.shape

In [ ]:
n_samples_to_plot = 2000
trapallada.plot.signals_array(signals.squeeze()[:, :n_samples_to_plot])

A signal to *upsample* signals

In [ ]:
def upsample_signals(signals: Float[Array, 'batch 1 time'], target_size: int) -> Float[Array, 'batch 1 time']:
    
    upsampling_factor = np.ceil(target_size / signals.shape[-1])
    
    return scipy.signal.resample_poly(signals, upsampling_factor, 1, axis=-1)[..., :target_size]

In [ ]:
# resampled_signals = upsample_signals(signals, 10_000)
# resampled_signals.shape

In [ ]:
# n_samples_to_plot = 2_000
# trapallada.plot.signals_array(resampled_signals[..., :n_samples_to_plot].squeeze())

In [ ]:
# resampled_signals[0].shape

In [ ]:
i = 12
x_pred, z_mean, z_sd = model(signals[i].numpy(), jr.PRNGKey(42))

# first dimension of `x_pred` is 1; below it's squeezed out
df = pd.DataFrame({'prediction': x_pred[0], 'actual': signals[i].ravel()})

trapallada.plot.df(df.iloc[:n_samples_to_plot])